# Guía de Laboratorio 1 — Gestión de Tareas y Semáforos en FreeRTOS

**Hardware:** STM32 Nucleo F103RB · 4 LEDs externos · 1 pulsador
**Librerías:** `FreeRTOS.h`, `task.h`, `semphr.h`
**Restricción:** Solo API nativa FreeRTOS + STM32 HAL. Sin CMSIS-RTOS.

```c
/* Pines — configurar en MX_GPIO_Init() vía STM32CubeMX */
#define LED1_PIN   GPIO_PIN_0
#define LED1_PORT  GPIOA
#define LED2_PIN   GPIO_PIN_1
#define LED2_PORT  GPIOA
#define LED3_PIN   GPIO_PIN_4
#define LED3_PORT  GPIOA
#define LED4_PIN   GPIO_PIN_5
#define LED4_PORT  GPIOA
#define BTN_PIN    GPIO_PIN_13
#define BTN_PORT   GPIOC
```

## Desafío 1 — Concurrencia Básica

**Objetivo:** Comprender la creación de tareas y la transición entre estados de ejecución.

**Restricción:** Las 4 tareas tienen la misma prioridad. **No usar** servicios bloqueantes de FreeRTOS (se permite `HAL_Delay`).

**Comportamiento:** Los 4 LEDs parpadean de forma independiente a 200 ms, 400 ms, 600 ms y 800 ms.

### Solución — Desafío 1

```c
#include "FreeRTOS.h"
#include "task.h"
#include "main.h"

static void vTarea_LED1(void *pvParams) {
    for (;;) { HAL_GPIO_TogglePin(LED1_PORT, LED1_PIN); HAL_Delay(200); }
}
static void vTarea_LED2(void *pvParams) {
    for (;;) { HAL_GPIO_TogglePin(LED2_PORT, LED2_PIN); HAL_Delay(400); }
}
static void vTarea_LED3(void *pvParams) {
    for (;;) { HAL_GPIO_TogglePin(LED3_PORT, LED3_PIN); HAL_Delay(600); }
}
static void vTarea_LED4(void *pvParams) {
    for (;;) { HAL_GPIO_TogglePin(LED4_PORT, LED4_PIN); HAL_Delay(800); }
}

int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    xTaskCreate(vTarea_LED1, "LED1", 128, NULL, 1, NULL);
    xTaskCreate(vTarea_LED2, "LED2", 128, NULL, 1, NULL);
    xTaskCreate(vTarea_LED3, "LED3", 128, NULL, 1, NULL);
    xTaskCreate(vTarea_LED4, "LED4", 128, NULL, 1, NULL);

    vTaskStartScheduler();
    for (;;);
}
```

### Análisis — Desafío 1

- **¿Todas las tareas se ejecutan?** Sí. El scheduler las alterna en round-robin al mismo nivel de prioridad.
- **¿Hay concurrencia real?** No es paralelismo verdadero (un solo core), pero el scheduler intercala la ejecución creando la ilusión de concurrencia.
- **¿Los tiempos son precisos?** **No.** `HAL_Delay()` es un busy-wait que bloquea **todo el sistema** sin ceder la CPU al scheduler. Cuando una tarea ejecuta `HAL_Delay(800)`, las demás no avanzan aunque estén listas. Con períodos de 20/40/60/80 ms el error relativo es notoriamente mayor porque el tiempo de conmutación de contexto es una fracción significativa del período.

## Desafío 2 — Instanciación Múltiple y Paso de Parámetros

**Objetivo:** Evitar duplicación de código con `pvParameters`. Usar `vTaskDelay` para ceder la CPU.

**Comportamiento:** Una única función `vTareaParpadeo` instanciada 4 veces, cada una con su configuración (puerto, pin, período) empaquetada en un `struct`.

### Solución — Desafío 2

```c
#include "FreeRTOS.h"
#include "task.h"
#include "main.h"

typedef struct {
    GPIO_TypeDef *puerto;
    uint16_t      pin;
    TickType_t    retardo_ms;
} LedConfig_t;

static void vTareaParpadeo(void *pvParams) {
    LedConfig_t *cfg = (LedConfig_t *)pvParams;
    for (;;) {
        HAL_GPIO_TogglePin(cfg->puerto, cfg->pin);
        vTaskDelay(pdMS_TO_TICKS(cfg->retardo_ms));
    }
}

/* Variables static: deben sobrevivir al scheduler, no pueden estar en el stack de main() */
static LedConfig_t led1 = {GPIOA, GPIO_PIN_0, 200};
static LedConfig_t led2 = {GPIOA, GPIO_PIN_1, 400};
static LedConfig_t led3 = {GPIOA, GPIO_PIN_4, 600};
static LedConfig_t led4 = {GPIOA, GPIO_PIN_5, 800};

int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    xTaskCreate(vTareaParpadeo, "LED1", 128, &led1, 1, NULL);
    xTaskCreate(vTareaParpadeo, "LED2", 128, &led2, 1, NULL);
    xTaskCreate(vTareaParpadeo, "LED3", 128, &led3, 1, NULL);
    xTaskCreate(vTareaParpadeo, "LED4", 128, &led4, 1, NULL);

    vTaskStartScheduler();
    for (;;);
}
```

### Análisis — Desafío 2

- **Diferencia clave con Desafío 1:** `vTaskDelay()` pone la tarea en estado **Blocked** e informa al scheduler que no necesita la CPU por N ticks. El scheduler puede ejecutar **otra tarea lista** durante ese tiempo: concurrencia real y eficiente.
- Los tiempos son **más precisos** porque el scheduler gestiona los wake-ups con el tick timer interno, sin busy-wait.
- La estructura `LedConfig_t` es `static`: el puntero pasado a la tarea debe seguir válido después de que `main()` llame a `vTaskStartScheduler()`; una variable local de `main()` quedaría en stack indefinido.

## Desafío 3 — Polling vs. Bloqueo Eficiente

**Objetivo:** Leer entradas físicas sin acaparar el CPU (evitar inanición de tareas de menor prioridad).

**Comportamiento:**
- LED1 → tarea de baja prioridad, parpadea a 500 ms con `HAL_Delay` (nunca se bloquea ante el scheduler).
- LED2 → tarea de alta prioridad, sigue el estado del pulsador por polling. **Tiempo de respuesta ≤ 10 ms.**

### Solución — Desafío 3

```c
#include "FreeRTOS.h"
#include "task.h"
#include "main.h"

/* Baja prioridad: busy-wait — puede ser desplazada por la tarea de botón */
static void vTarea_LED1(void *pvParams) {
    for (;;) {
        HAL_GPIO_TogglePin(LED1_PORT, LED1_PIN);
        HAL_Delay(500);
    }
}

/* Alta prioridad: polling del botón con vTaskDelay de 5 ms (≤ deadline de 10 ms) */
static void vTarea_Boton(void *pvParams) {
    for (;;) {
        GPIO_PinState estado = HAL_GPIO_ReadPin(BTN_PORT, BTN_PIN);
        HAL_GPIO_WritePin(LED2_PORT, LED2_PIN,
            (estado == GPIO_PIN_RESET) ? GPIO_PIN_SET : GPIO_PIN_RESET);
        vTaskDelay(pdMS_TO_TICKS(5));
    }
}

int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    xTaskCreate(vTarea_LED1,  "LED1", 128, NULL, 1, NULL);  /* Prioridad baja  */
    xTaskCreate(vTarea_Boton, "BTN",  128, NULL, 2, NULL);  /* Prioridad alta  */

    vTaskStartScheduler();
    for (;;);
}
```

### Análisis — Desafío 3

- **Si se omite `vTaskDelay()` en la tarea de botón:** la tarea de mayor prioridad nunca cede la CPU → LED1 deja de parpadear (**starvation**).
- **Efecto al aumentar el delay:** la latencia de respuesta del LED2 aumenta proporcionalmente. Con 5 ms se cumple el deadline de 10 ms con margen.
- **Tiempo máximo que satisface el deadline:** ≤ 10 ms. Se recomienda 5 ms para absorber el jitter del scheduler.

## Desafío 4 — Periodicidad Absoluta vs. Relativa (Time Drift)

**Objetivo:** Demostrar la diferencia entre retardos relativos (`vTaskDelay`) y absolutos (`xTaskDelayUntil`).

**Comportamiento:** Ambas tareas intentan hacer parpadear un LED cada 500 ms exactos, pero cada iteración incluye una carga de CPU simulada de 100 ms (`HAL_Delay(100)`).

### Solución — Desafío 4

```c
#include "FreeRTOS.h"
#include "task.h"
#include "main.h"

/* Tarea A: retardo RELATIVO — acumula deriva con el tiempo */
static void vTareaA(void *pvParams) {
    for (;;) {
        HAL_Delay(100);                        /* Carga simulada: 100 ms         */
        HAL_GPIO_TogglePin(LED1_PORT, LED1_PIN);
        vTaskDelay(pdMS_TO_TICKS(400));        /* 400 ms + 100 ms carga ≈ 500 ms */
    }
    /*
     * Período efectivo real: 100 ms (carga) + 400 ms (delay) + overhead scheduler
     * → mayor de 500 ms → deriva acumulativa observable
     */
}

/* Tarea B: retardo ABSOLUTO — sin deriva, período exactamente 500 ms */
static void vTareaB(void *pvParams) {
    TickType_t xLastWakeTime = xTaskGetTickCount();
    for (;;) {
        HAL_Delay(100);                        /* Carga simulada: 100 ms         */
        HAL_GPIO_TogglePin(LED2_PORT, LED2_PIN);
        xTaskDelayUntil(&xLastWakeTime, pdMS_TO_TICKS(500));
        /*
         * xTaskDelayUntil descuenta el tiempo ya transcurrido desde xLastWakeTime.
         * Si la carga tomó 100 ms, solo espera 400 ms más → período = 500 ms exactos.
         */
    }
}

int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    xTaskCreate(vTareaA, "TaskA", 128, NULL, 1, NULL);
    xTaskCreate(vTareaB, "TaskB", 128, NULL, 1, NULL);

    vTaskStartScheduler();
    for (;;);
}
```

### Análisis — Desafío 4

- **Los argumentos NO son iguales.** Para `vTaskDelay` se usa **400 ms** (500 ms target − 100 ms de carga). Para `xTaskDelayUntil` se usa **500 ms** (el período absoluto deseado; la API descuenta el tiempo transcurrido automáticamente).
- **Si ambos fueran 500 ms:** la Tarea A tendría período efectivo ≈ 600 ms (100 ms carga + 500 ms delay + overhead), derivando progresivamente. La Tarea B seguiría exacta.
- `xTaskDelayUntil` elimina la **deriva acumulativa**: el wake-up siempre ocurre en múltiplos exactos del período desde el punto de partida, independientemente de cuánto tarde la ejecución del cuerpo de la tarea.

## Desafío 5 — Control de Prioridades Dinámico

**Objetivo:** Observar el scheduling preemptivo cuando la prioridad cambia en tiempo de ejecución.

**Comportamiento:**
- Tarea A (prioridad 1): parpadea LED1 cada 300 ms.
- Tarea B (prioridad 1): parpadea LED2 cada 300 ms y monitorea el botón.
- Al presionar el botón, Tarea A sube su prioridad en 1 durante 3 segundos (LED2 se congela). Luego restituye.

### Solución — Desafío 5

```c
#include "FreeRTOS.h"
#include "task.h"
#include "main.h"

static TaskHandle_t xTareaA_Handle = NULL;

static void vTareaA(void *pvParams) {
    for (;;) {
        HAL_GPIO_TogglePin(LED1_PORT, LED1_PIN);
        vTaskDelay(pdMS_TO_TICKS(300));
    }
}

static void vTareaB(void *pvParams) {
    for (;;) {
        HAL_GPIO_TogglePin(LED2_PORT, LED2_PIN);
        vTaskDelay(pdMS_TO_TICKS(300));

        if (HAL_GPIO_ReadPin(BTN_PORT, BTN_PIN) == GPIO_PIN_RESET) {
            UBaseType_t uxPrio = uxTaskPriorityGet(xTareaA_Handle);
            vTaskPrioritySet(xTareaA_Handle, uxPrio + 1);  /* A pasa a prio 2 */
            vTaskDelay(pdMS_TO_TICKS(3000));                /* Espera 3 s       */
            vTaskPrioritySet(xTareaA_Handle, uxPrio);       /* Restituye prio 1 */
            /* Anti-rebote: espera a que se suelte el botón */
            while (HAL_GPIO_ReadPin(BTN_PORT, BTN_PIN) == GPIO_PIN_RESET)
                vTaskDelay(pdMS_TO_TICKS(10));
        }
    }
}

int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    xTaskCreate(vTareaA, "A", 128, NULL, 1, &xTareaA_Handle);
    xTaskCreate(vTareaB, "B", 128, NULL, 1, NULL);

    vTaskStartScheduler();
    for (;;);
}
```

### Análisis — Desafío 5

- **Al restablecer la prioridad:** el scheduler vuelve a ejecutar ambas en round-robin. LED2 retoma su parpadeo de inmediato.
- **Conteo de 3 s:** se hace con `vTaskDelay(3000 ms)` dentro de la Tarea B. Durante esos 3 s, B está **Blocked**, por lo que no interfiere con A (que es de mayor prioridad).
- **Problema típico en primer intento:** si no se espera a que se suelte el botón, la condición se re-dispara al regresar del `vTaskDelay`, elevando la prioridad de nuevo inmediatamente.
- **`uxTaskPriorityGet(NULL)`** devuelve la prioridad de la tarea que llama; con el handle de otra tarea, devuelve la de esa tarea.

## Desafío 6 — Sincronización Unidireccional (Deferred Interrupt Processing)

**Objetivo:** Reemplazar el polling por espera pasiva usando semáforos binarios e interrupciones EXTI.

**Comportamiento:** Una tarea bloqueada (0% CPU) despierta al presionar el botón (la ISR da el semáforo), conmuta LED3 y vuelve a bloquearse.

### Solución — Desafío 6

```c
#include "FreeRTOS.h"
#include "task.h"
#include "semphr.h"
#include "main.h"

static SemaphoreHandle_t xSemBtn = NULL;

/* ISR — BTN_PIN configurado en STM32CubeMX como EXTI, flanco descendente */
void HAL_GPIO_EXTI_Callback(uint16_t GPIO_Pin) {
    if (GPIO_Pin == BTN_PIN) {
        BaseType_t xHPTW = pdFALSE;
        xSemaphoreGiveFromISR(xSemBtn, &xHPTW);
        portYIELD_FROM_ISR(xHPTW);  /* Fuerza cambio de contexto si despertó tarea de mayor prio */
    }
}

/* Tarea bloqueada: 0% CPU mientras espera la pulsación */
static void vTarea_LED3(void *pvParams) {
    for (;;) {
        xSemaphoreTake(xSemBtn, portMAX_DELAY);
        HAL_GPIO_TogglePin(LED3_PORT, LED3_PIN);
    }
}

/* Tarea auxiliar de baja prioridad: evidencia de CPU libre */
static void vTarea_LED1(void *pvParams) {
    for (;;) {
        HAL_GPIO_TogglePin(LED1_PORT, LED1_PIN);
        vTaskDelay(pdMS_TO_TICKS(200));
    }
}

int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();   /* BTN_PIN → EXTI con interrupción habilitada en NVIC */

    xSemBtn = xSemaphoreCreateBinary();

    xTaskCreate(vTarea_LED3, "LED3", 128, NULL, 2, NULL);  /* Alta prio  */
    xTaskCreate(vTarea_LED1, "LED1", 128, NULL, 1, NULL);  /* Baja prio  */

    vTaskStartScheduler();
    for (;;);
}
```

### Análisis — Desafío 6

- **Evidencia de no consumo de CPU:** LED1 (baja prioridad) parpadea constantemente. Si la tarea de LED3 estuviera en polling activo sin ceder la CPU, consumiría todo el tiempo de su prioridad y LED1 quedaría congelado. Su parpadeo continuo demuestra que LED3 **no monopoliza la CPU**.
- No se crean nuevas tareas: LED1 ya es parte del sistema y actúa como indicador visual.
- **`portYIELD_FROM_ISR`** es esencial: sin él el cambio de contexto ocurriría recién en el próximo tick del scheduler (hasta 1 ms de latencia extra).

## Desafío 7 — Secuenciamiento de Tareas (Task Synchronization)

**Objetivo:** Imponer un orden de ejecución estricto con semáforos binarios como señales de habilitación.

**Comportamiento:** 3 tareas encienden sus LEDs en orden 1 → 2 → 3 → 1 … apagando el anterior en cada paso.

### Solución — Desafío 7

```c
#include "FreeRTOS.h"
#include "task.h"
#include "semphr.h"
#include "main.h"

static SemaphoreHandle_t xSem12 = NULL;  /* Tarea 1 habilita Tarea 2 */
static SemaphoreHandle_t xSem23 = NULL;  /* Tarea 2 habilita Tarea 3 */
static SemaphoreHandle_t xSem31 = NULL;  /* Tarea 3 habilita Tarea 1 */

static void vTarea1(void *pvParams) {
    for (;;) {
        xSemaphoreTake(xSem31, portMAX_DELAY);
        HAL_GPIO_WritePin(LED3_PORT, LED3_PIN, GPIO_PIN_RESET); /* Apaga LED3 */
        HAL_GPIO_WritePin(LED1_PORT, LED1_PIN, GPIO_PIN_SET);   /* Enciende LED1 */
        vTaskDelay(pdMS_TO_TICKS(500));
        xSemaphoreGive(xSem12);
    }
}

static void vTarea2(void *pvParams) {
    for (;;) {
        xSemaphoreTake(xSem12, portMAX_DELAY);
        HAL_GPIO_WritePin(LED1_PORT, LED1_PIN, GPIO_PIN_RESET);
        HAL_GPIO_WritePin(LED2_PORT, LED2_PIN, GPIO_PIN_SET);
        vTaskDelay(pdMS_TO_TICKS(500));
        xSemaphoreGive(xSem23);
    }
}

static void vTarea3(void *pvParams) {
    for (;;) {
        xSemaphoreTake(xSem23, portMAX_DELAY);
        HAL_GPIO_WritePin(LED2_PORT, LED2_PIN, GPIO_PIN_RESET);
        HAL_GPIO_WritePin(LED3_PORT, LED3_PIN, GPIO_PIN_SET);
        vTaskDelay(pdMS_TO_TICKS(500));
        xSemaphoreGive(xSem31);
    }
}

int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    xSem12 = xSemaphoreCreateBinary();
    xSem23 = xSemaphoreCreateBinary();
    xSem31 = xSemaphoreCreateBinary();

    xTaskCreate(vTarea1, "T1", 128, NULL, 1, NULL);
    xTaskCreate(vTarea2, "T2", 128, NULL, 1, NULL);
    xTaskCreate(vTarea3, "T3", 128, NULL, 1, NULL);

    xSemaphoreGive(xSem31);   /* Arranque: habilita Tarea 1 */

    vTaskStartScheduler();
    for (;;);
}
```

### Análisis — Desafío 7

- **Para alterar el orden:** cambiar qué semáforo toma y da cada tarea reordena la secuencia sin tocar la lógica interna.
- **Implementación con una única función:** es posible pasando como parámetros `{semáforo_a_tomar, semáforo_a_dar, led_propio, led_anterior}`. Eso evita duplicar la estructura de take/enciende/delay/give.

## Desafío 8 — Arbitraje de Recursos Compartidos (Mutex)

**Objetivo:** Garantizar acceso exclusivo a los LEDs para que las secuencias visuales no se corrompan.

**Comportamiento:**
- Tarea A: secuencia rápida 1 → 2 → 3 → 4.
- Tarea B: parpadeo total × 4 (todos ON / todos OFF).
- Ambas compiten por el Mutex; quien lo obtiene completa su secuencia entera antes de soltarlo.

### Solución — Desafío 8

```c
#include "FreeRTOS.h"
#include "task.h"
#include "semphr.h"
#include "main.h"

static SemaphoreHandle_t xMutexLEDs = NULL;

static const uint16_t  leds_pin[]  = {LED1_PIN, LED2_PIN, LED3_PIN, LED4_PIN};
static GPIO_TypeDef   *leds_port[] = {LED1_PORT, LED2_PORT, LED3_PORT, LED4_PORT};

/* Tarea A: secuencia 1 → 2 → 3 → 4 */
static void vTareaA(void *pvParams) {
    for (;;) {
        xSemaphoreTake(xMutexLEDs, portMAX_DELAY);
        for (int i = 0; i < 4; i++) {
            HAL_GPIO_WritePin(leds_port[i], leds_pin[i], GPIO_PIN_SET);
            HAL_Delay(100);    /* HAL_Delay dentro del Mutex: no cede CPU ni el Mutex */
            HAL_GPIO_WritePin(leds_port[i], leds_pin[i], GPIO_PIN_RESET);
        }
        xSemaphoreGive(xMutexLEDs);
    }
}

/* Tarea B: todos ON / todos OFF × 4 */
static void vTareaB(void *pvParams) {
    for (;;) {
        xSemaphoreTake(xMutexLEDs, portMAX_DELAY);
        for (int rep = 0; rep < 4; rep++) {
            for (int i = 0; i < 4; i++)
                HAL_GPIO_WritePin(leds_port[i], leds_pin[i], GPIO_PIN_SET);
            HAL_Delay(200);
            for (int i = 0; i < 4; i++)
                HAL_GPIO_WritePin(leds_port[i], leds_pin[i], GPIO_PIN_RESET);
            HAL_Delay(200);
        }
        xSemaphoreGive(xMutexLEDs);
    }
}

int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    xMutexLEDs = xSemaphoreCreateMutex();

    xTaskCreate(vTareaA, "A", 128, NULL, 1, NULL);
    xTaskCreate(vTareaB, "B", 128, NULL, 1, NULL);

    vTaskStartScheduler();
    for (;;);
}
```

### Análisis — Desafío 8

- **¿`vTaskDelay` o `HAL_Delay` dentro de la sección crítica?** Se usa **`HAL_Delay`** (busy-wait). Si se usara `vTaskDelay()`, la tarea cedería la CPU **mientras mantiene el Mutex**: la otra tarea intentaría tomarlo, quedaría bloqueada, y el CPU estaría ocioso con el recurso tomado sin progreso útil. Este es el **anti-patrón "tener el Mutex dormido"**.
- **Si se reescribe con `vTaskDelay`:** el efecto visual es idéntico (el Mutex garantiza la secuencia completa igual), pero el sistema es menos eficiente: más cambios de contexto innecesarios y mayor latencia para la tarea que espera.
- El Mutex de FreeRTOS soporta **herencia de prioridad**: si una tarea de alta prioridad espera el Mutex, la tarea que lo tiene sube temporalmente su prioridad para evitar inversión de prioridad.

## Desafío 9 — Suspensión y Reanudación de Tareas

**Objetivo:** Controlar el estado de ejecución de una tarea externa con `vTaskSuspend` / `vTaskResume`.

**Comportamiento:** Tarea A genera un patrón de luces en los 4 LEDs. Primera pulsación → suspende A (LEDs congelados). Segunda pulsación → reanuda A.

### Solución — Desafío 9

```c
#include "FreeRTOS.h"
#include "task.h"
#include "semphr.h"
#include "main.h"

static TaskHandle_t      xTareaA_Handle = NULL;
static SemaphoreHandle_t xSemBtn        = NULL;

/* ISR — mejor tiempo de respuesta que polling */
void HAL_GPIO_EXTI_Callback(uint16_t GPIO_Pin) {
    if (GPIO_Pin == BTN_PIN) {
        BaseType_t xHPTW = pdFALSE;
        xSemaphoreGiveFromISR(xSemBtn, &xHPTW);
        portYIELD_FROM_ISR(xHPTW);
    }
}

/* Tarea A: patrón de luces secuencial */
static void vTareaA(void *pvParams) {
    const uint16_t  pines[]   = {LED1_PIN, LED2_PIN, LED3_PIN, LED4_PIN};
    GPIO_TypeDef   *puertos[] = {LED1_PORT, LED2_PORT, LED3_PORT, LED4_PORT};
    int i = 0;
    for (;;) {
        for (int j = 0; j < 4; j++)
            HAL_GPIO_WritePin(puertos[j], pines[j], GPIO_PIN_RESET);
        HAL_GPIO_WritePin(puertos[i], pines[i], GPIO_PIN_SET);
        i = (i + 1) % 4;
        vTaskDelay(pdMS_TO_TICKS(300));
    }
}

/* Tarea B: alterna suspend / resume en cada pulsación */
static void vTareaB(void *pvParams) {
    uint8_t suspendida = 0;
    for (;;) {
        xSemaphoreTake(xSemBtn, portMAX_DELAY);
        if (!suspendida) {
            vTaskSuspend(xTareaA_Handle);
            suspendida = 1;
        } else {
            vTaskResume(xTareaA_Handle);
            suspendida = 0;
        }
        vTaskDelay(pdMS_TO_TICKS(200));   /* Anti-rebote */
    }
}

int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    xSemBtn = xSemaphoreCreateBinary();

    xTaskCreate(vTareaA, "A", 128, NULL, 1, &xTareaA_Handle);
    xTaskCreate(vTareaB, "B", 128, NULL, 2, NULL);

    vTaskStartScheduler();
    for (;;);
}
```

### Análisis — Desafío 9

- **Polling vs. Interrupción:** La ISR + semáforo ofrece latencia submilisegundo. Con polling la latencia máxima es el período del `vTaskDelay` de la tarea de botón.
- **¿Por qué ISR es más segura?** La ISR responde en el instante del flanco; el polling puede perder pulsaciones muy cortas si el delay de la tarea supera la duración del pulso.
- **`xSemaphoreGiveFromISR` + `portYIELD_FROM_ISR`:** si se llamara `xSemaphoreGive()` directamente desde la ISR se corrompería el estado interno del scheduler (función no reentrant en contexto de interrupción).

## Desafío 10 — Autodestrucción y Gestión de Memoria

**Objetivo:** Comprender la liberación de recursos y el ciclo de vida final de una tarea.

**Comportamiento:** Al iniciar, una tarea ejecuta un Light Show (animación) durante 10 s. Al terminar, enciende LED4 permanentemente y se elimina con `vTaskDelete(NULL)` para liberar su stack.

### Solución — Desafío 10

```c
#include "FreeRTOS.h"
#include "task.h"
#include "main.h"

static void vTarea_LightShow(void *pvParams) {
    const uint16_t  pines[]   = {LED1_PIN, LED2_PIN, LED3_PIN, LED4_PIN};
    GPIO_TypeDef   *puertos[] = {LED1_PORT, LED2_PORT, LED3_PORT, LED4_PORT};
    TickType_t xInicio = xTaskGetTickCount();
    int pos = 0, dir = 1;

    /* Animación tipo "bouncing ball" durante 10 s */
    while ((xTaskGetTickCount() - xInicio) < pdMS_TO_TICKS(10000)) {
        for (int j = 0; j < 4; j++)
            HAL_GPIO_WritePin(puertos[j], pines[j], GPIO_PIN_RESET);
        HAL_GPIO_WritePin(puertos[pos], pines[pos], GPIO_PIN_SET);
        pos += dir;
        if (pos >= 3) dir = -1;
        if (pos <= 0) dir =  1;
        vTaskDelay(pdMS_TO_TICKS(150));
    }

    /* Fin de animación */
    for (int j = 0; j < 4; j++)
        HAL_GPIO_WritePin(puertos[j], pines[j], GPIO_PIN_RESET);
    HAL_GPIO_WritePin(LED4_PORT, LED4_PIN, GPIO_PIN_SET);   /* LED4 fijo: evidencia */

    vTaskDelete(NULL);   /* Libera el stack y elimina esta tarea del scheduler */
}

int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    xTaskCreate(vTarea_LightShow, "SHOW", 256, NULL, 1, NULL);

    vTaskStartScheduler();
    for (;;);
}
```

### Análisis — Desafío 10

- **Evidencia visible de eliminación:** LED4 queda encendido de forma fija y la animación cesa. Como la tarea ya no existe en el scheduler, ningún código la reactiva.
- **Liberación de memoria:** el stack (256 words ≈ 1 KB en STM32) regresa al heap de FreeRTOS y puede ser aprovechado por `xTaskCreate` futuras.
- **Forma adicional de verificarlo:** llamar `uxTaskGetNumberOfTasks()` antes y después del `vTaskDelete` y mostrar el resultado (por UART o mediante el encendido de otro LED).